In [1]:
import time

import numpy as np

from pybandits.model import (
    BayesianLogisticRegression,
    BayesianNeuralNetwork,
    LegacyBayesianLogisticRegression,
    StudentT,
)

%load_ext autoreload
%autoreload 2

## Parameters

In [3]:
n_features = 20
n_samples = 100000
bnn_dims = [n_features, 10, 5, 1]

## Generate data

In [4]:
context = np.random.randn(n_samples, n_features)
label = np.random.binomial(1, 0.5, n_samples).tolist()
print(f"context shape: {context.shape}")

context shape: (100000, 20)


## Create models

In [6]:
new_blr = BayesianLogisticRegression(
    alpha=StudentT(), betas=[StudentT() for _ in range(n_features)], update_method="VI"
)
old_blr = LegacyBayesianLogisticRegression(
    alpha=StudentT(), betas=[StudentT() for _ in range(n_features)], update_method="VI"
)
bnn = BayesianNeuralNetwork.cold_start(dim_list=bnn_dims, update_method="VI")

# Compare times for sample_proba/update

## Compare sample_proba

In [7]:
%%timeit
old_blr.sample_proba(context)

236 ms ± 6.11 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
%%timeit
new_blr.sample_proba(context)

Sampling: [b0, out, w0]
Sampling: [b0, out, w0]
Sampling: [b0, out, w0]
Sampling: [b0, out, w0]
Sampling: [b0, out, w0]
Sampling: [b0, out, w0]
Sampling: [b0, out, w0]
Sampling: [b0, out, w0]


221 ms ± 6.03 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [9]:
%%timeit
bnn.sample_proba(context)

Sampling: [b0, b1, b2, b3, out, w0, w1, w2, w3]
Sampling: [b0, b1, b2, b3, out, w0, w1, w2, w3]
Sampling: [b0, b1, b2, b3, out, w0, w1, w2, w3]
Sampling: [b0, b1, b2, b3, out, w0, w1, w2, w3]
Sampling: [b0, b1, b2, b3, out, w0, w1, w2, w3]
Sampling: [b0, b1, b2, b3, out, w0, w1, w2, w3]
Sampling: [b0, b1, b2, b3, out, w0, w1, w2, w3]
Sampling: [b0, b1, b2, b3, out, w0, w1, w2, w3]


2 s ± 19.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## comapre update

In [10]:
curr_time = time.time()
old_blr.update(context, label)
print(f"Old BLR took {time.time() - curr_time} seconds")

Finished [100%]: Average Loss = 73,738


Old BLR took 366.5583846569061 seconds


In [11]:
curr_time = time.time()
new_blr.update(context, label)
print(f"NEW BLR took {time.time() - curr_time} seconds")

Interrupted at 2,854 [28%]: Average Loss = 1.2412e+05


KeyboardInterrupt: 

In [ ]:
curr_time = time.time()
bnn.update(context, label)
print(f"BNN took {time.time() - curr_time} seconds")

Finished [100%]: Average Loss = 70,139


BNN took 1496.9373173713684 seconds
